In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
sns.set_style('whitegrid')

df = pd.read_parquet('../data/loans_clean.parquet')
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")
print(f"Default rate: {df['default'].mean():.2%}")

Loaded 1,345,310 rows and 105 columns
Default rate: 19.96%


In [2]:
categoricals = ['term', 'home_ownership', 'verification_status', 'purpose', 'application_type']

for col in categoricals:
    print(f"\n{'='*50}\n{col}\n{'='*50}")
    stats = df.groupby(col).agg(
        n_loans=('default', 'size'),
        default_rate=('default', 'mean')
    ).sort_values('default_rate', ascending=False).round(4)
    print(stats)


term
           n_loans  default_rate
term                            
60 months   324567        0.3245
36 months  1020743        0.1599

home_ownership
                n_loans  default_rate
home_ownership                       
RENT             534421        0.2322
OWN              144832        0.2062
ANY                 286        0.1958
OTHER               144        0.1875
MORTGAGE         665579        0.1721
NONE                 48        0.1458

verification_status
                     n_loans  default_rate
verification_status                       
Verified              418336        0.2385
Source Verified       521273        0.2095
Not Verified          405701        0.1467

purpose
                    n_loans  default_rate
purpose                                  
small_business        15416        0.2971
renewable_energy        933        0.2369
moving                 9480        0.2335
house                  7253        0.2188
medical               15554        0.2178
deb

In [3]:
numerics = ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 
            'fico_range_low', 'open_acc', 'revol_util', 'credit_history_months']

print(df[numerics].describe(percentiles=[0.01, 0.5, 0.99]).round(2))

        loan_amnt    int_rate  installment   annual_inc         dti  \
count  1345310.00  1345310.00   1345310.00   1345310.00  1344936.00   
mean     14419.97       13.24       438.08     76247.64       18.28   
std       8717.05        4.77       261.51     69925.10       11.16   
min        500.00        5.31         4.93         0.00       -1.00   
1%        1500.00        5.32        52.86     18000.00        1.78   
50%      12000.00       12.74       375.43     65000.00       17.61   
99%      35000.00       26.30      1221.50    250000.00       38.47   
max      40000.00       30.99      1719.83  10999200.00      999.00   

       fico_range_low    open_acc  revol_util  credit_history_months  
count      1345310.00  1345310.00  1344453.00              1345310.0  
mean           696.19       11.59       51.81                 195.11  
std             31.85        5.47       24.52                  90.07  
min            625.00        0.00        0.00                   36.0  
1%   

In [6]:
# Convert sentinel values to NaN
df['dti'] = df['dti'].replace([-1, 999], np.nan)
df['credit_history_months'] = df['credit_history_months'].replace(999, np.nan)

# Cap extreme outliers
df['annual_inc'] = df['annual_inc'].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=150)
df['dti'] = df['dti'].clip(upper=100)  

# Re-check the cleaned features
print(df[['annual_inc', 'dti', 'revol_util', 'credit_history_months']].describe(
    percentiles=[0.01, 0.5, 0.99]
).round(2))

       annual_inc         dti  revol_util  credit_history_months
count  1345310.00  1344896.00  1344453.00              1345309.0
mean     75911.59       18.21       51.80                 195.11
std      51807.99        8.82       24.48                  90.07
min          0.00        0.00        0.00                   36.0
1%       18000.00        1.78        1.10                   46.0
50%      65000.00       17.61       52.20                  177.0
99%     250000.00       38.46       98.20                  478.0
max    1000000.00      100.00      100.00                  852.0


In [7]:
def default_by_buckets(df, col, n_buckets=10):
    buckets = pd.qcut(df[col], n_buckets, duplicates='drop')
    return df.groupby(buckets, observed=True).agg(
        n_loans=('default', 'size'),
        default_rate=('default', 'mean')
    ).round(4)

for col in ['fico_range_low', 'dti', 'annual_inc', 'int_rate']:
    print(f"\n{'='*60}\nDefault rate by {col} (deciles)\n{'='*60}")
    print(default_by_buckets(df, col))


Default rate by fico_range_low (deciles)
                  n_loans  default_rate
fico_range_low                         
(624.999, 665.0]   237685        0.2628
(665.0, 670.0]     117202        0.2461
(670.0, 675.0]     104593        0.2378
(675.0, 680.0]     103340        0.2309
(680.0, 690.0]     178886        0.2115
(690.0, 695.0]      79100        0.1993
(695.0, 705.0]     138161        0.1826
(705.0, 720.0]     151648        0.1563
(720.0, 740.0]     110814        0.1312
(740.0, 845.0]     123881        0.0925

Default rate by dti (deciles)
                n_loans  default_rate
dti                                  
(-0.001, 7.27]   134671        0.1465
(7.27, 10.49]    134694        0.1526
(10.49, 13.02]   134581        0.1623
(13.02, 15.32]   134123        0.1731
(15.32, 17.61]   134413        0.1848
(17.61, 19.98]   134611        0.1971
(19.98, 22.6]    134721        0.2108
(22.6, 25.69]    134364        0.2283
(25.69, 29.78]   134448        0.2509
(29.78, 100.0]   134270      

In [8]:
# Compute missingness and default rate by missingness
missing_pct = df.isnull().mean().sort_values(ascending=False)
high_missing = missing_pct[missing_pct > 0.05]  # features with >5% missing

print(f"Features with >5% missing: {len(high_missing)}")
print("\nTop missing features:")
print((high_missing * 100).round(1).to_frame('pct_missing').head(20))

Features with >5% missing: 60

Top missing features:
                                     pct_missing
sec_app_mths_since_last_major_derog         99.5
sec_app_revol_util                          98.6
revol_bal_joint                             98.6
sec_app_fico_range_low                      98.6
sec_app_collections_12_mths_ex_med          98.6
sec_app_chargeoff_within_12_mths            98.6
sec_app_num_rev_accts                       98.6
sec_app_open_act_il                         98.6
sec_app_open_acc                            98.6
sec_app_mort_acc                            98.6
sec_app_inq_last_6mths                      98.6
sec_app_earliest_cr_line                    98.6
sec_app_fico_range_high                     98.6
verification_status_joint                   98.1
dti_joint                                   98.1
annual_inc_joint                            98.1
mths_since_last_record                      83.0
mths_since_recent_bc_dlq                    76.3
mths_since_last_

In [9]:
informative_missing_candidates = [
    'mths_since_last_delinq',
    'mths_since_last_record', 
    'mths_since_last_major_derog',
    'mths_since_recent_bc_dlq',
    'mths_since_recent_revol_delinq',
    'mths_since_recent_inq',
]

print("Default rate: missing vs present")
print("="*60)
for col in informative_missing_candidates:
    if col in df.columns:
        missing_mask = df[col].isnull()
        rate_missing = df.loc[missing_mask, 'default'].mean()
        rate_present = df.loc[~missing_mask, 'default'].mean()
        pct_missing = missing_mask.mean() * 100
        print(f"\n{col}")
        print(f"  Missing ({pct_missing:5.1f}%): default rate {rate_missing:.4f}")
        print(f"  Present ({100-pct_missing:5.1f}%): default rate {rate_present:.4f}")
        print(f"  Difference:                {rate_missing - rate_present:+.4f}")

Default rate: missing vs present

mths_since_last_delinq
  Missing ( 50.5%): default rate 0.1926
  Present ( 49.5%): default rate 0.2068
  Difference:                -0.0141

mths_since_last_record
  Missing ( 83.0%): default rate 0.1939
  Present ( 17.0%): default rate 0.2274
  Difference:                -0.0334

mths_since_last_major_derog
  Missing ( 73.7%): default rate 0.1926
  Present ( 26.3%): default rate 0.2194
  Difference:                -0.0268

mths_since_recent_bc_dlq
  Missing ( 76.3%): default rate 0.1965
  Present ( 23.7%): default rate 0.2098
  Difference:                -0.0134

mths_since_recent_revol_delinq
  Missing ( 66.6%): default rate 0.1960
  Present ( 33.4%): default rate 0.2068
  Difference:                -0.0107

mths_since_recent_inq
  Missing ( 12.9%): default rate 0.1488
  Present ( 87.1%): default rate 0.2072
  Difference:                -0.0584


In [10]:
# Numeric features only (correlation needs numbers)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Drop the target so we don't see correlations with default itself
numeric_cols = [c for c in numeric_cols if c not in ['default', 'issue_year']]

corr = df[numeric_cols].corr().abs()

# Get pairs above 0.9 correlation (highly redundant)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
high_corr_pairs.columns = ['feature_1', 'feature_2', 'abs_correlation']
print(high_corr_pairs.round(3))

                   feature_1                       feature_2  abs_correlation
0     sec_app_fico_range_low         sec_app_fico_range_high            1.000
1             fico_range_low                 fico_range_high            1.000
2                  loan_amnt                     funded_amnt            1.000
3                funded_amnt                 funded_amnt_inv            0.999
4                   open_acc                        num_sats            0.999
5                  loan_amnt                 funded_amnt_inv            0.999
6            num_actv_rev_tl             num_rev_tl_bal_gt_0            0.982
7                tot_cur_bal                 tot_hi_cred_lim            0.973
8               total_bal_il      total_il_high_credit_limit            0.957
9                funded_amnt                     installment            0.954
10                 loan_amnt                     installment            0.953
11           funded_amnt_inv                     installment    